# Physics-Informed Neural Network for the 1D Schrödinger Equation

This notebook implements a **Physics-Informed Neural Network (PINN)** to solve the **time-independent Schrödinger equation** for simple 1D quantum systems.

We solve:

$$
-\frac{1}{2}\frac{d^2\psi}{dx^2} + V(x)\psi(x) = E\psi(x)
$$

using a neural network that learns:

- the wavefunction $\psi(x)$
- the energy eigenvalue $E$

Potentials included:

- Particle in a box
- Harmonic oscillator
- Finite potential well


## 1. Install / import libraries

Run the next cell in Google Colab if needed.


In [ ]:
# Uncomment if needed in Colab
# !pip install torch matplotlib numpy


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


## 2. Problem setup

We use dimensionless units with $\hbar = 1$ and $m = 1$.

Change `POTENTIAL_TYPE` to one of:

- `'box'`
- `'harmonic'`
- `'finite_well'`


In [ ]:
# Choose potential
POTENTIAL_TYPE = 'harmonic'   # 'box', 'harmonic', 'finite_well'

# Domain
x_min, x_max = -5.0, 5.0
N_collocation = 400
N_plot = 1000

# Number of eigenstates to compute
N_STATES = 3

# Training hyperparameters
EPOCHS = 5000
LR = 1e-3

# Loss weights
LAMBDA_BC = 10.0
LAMBDA_NORM = 10.0
LAMBDA_ORTH = 20.0

# Finite well parameters
V0 = 5.0
a = 2.0


## 3. Define the potentials

In [ ]:
def potential(x, potential_type='harmonic'):
    if potential_type == 'box':
        return torch.zeros_like(x)

    elif potential_type == 'harmonic':
        return 0.5 * x**2

    elif potential_type == 'finite_well':
        return torch.where(torch.abs(x) <= a/2,
                           torch.zeros_like(x),
                           V0 * torch.ones_like(x))
    else:
        raise ValueError('Unknown potential type')


## 4. PINN model

The network outputs the wavefunction $\psi(x)$ and contains a trainable parameter for the energy $E$.


In [ ]:
class PINNWaveFunction(nn.Module):
    def __init__(self, hidden_dim=64, n_hidden=4):
        super().__init__()
        layers = []
        layers.append(nn.Linear(1, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

        self.energy = nn.Parameter(torch.tensor([1.0], dtype=torch.float32))

    def forward_raw(self, x):
        return self.net(x)

    def forward(self, x, potential_type='harmonic'):
        raw = self.forward_raw(x)

        if potential_type == 'box':
            xi = (x - x_min) / (x_max - x_min)
            envelope = xi * (1.0 - xi)
            return envelope * raw

        elif potential_type == 'harmonic':
            envelope = torch.exp(-0.1 * x**2)
            return envelope * raw

        elif potential_type == 'finite_well':
            envelope = torch.exp(-0.05 * x**2)
            return envelope * raw

        return raw


## 5. Helper functions

In [ ]:
def second_derivative(y, x):
    dy_dx = torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True
    )[0]

    d2y_dx2 = torch.autograd.grad(
        dy_dx, x,
        grad_outputs=torch.ones_like(dy_dx),
        create_graph=True,
        retain_graph=True
    )[0]
    return d2y_dx2


def trapz_torch(y, x):
    y = y.squeeze()
    x = x.squeeze()
    return torch.trapz(y, x)


def normalize_wavefunction(psi, x):
    norm = torch.sqrt(trapz_torch(psi**2, x) + 1e-12)
    return psi / norm


def bc_loss(model, potential_type):
    xb = torch.tensor([[x_min], [x_max]], dtype=torch.float32, device=device)
    psi_b = model(xb, potential_type=potential_type)
    return torch.mean(psi_b**2)


def normalization_loss(psi, x):
    prob = psi**2
    norm_val = trapz_torch(prob, x)
    return (norm_val - 1.0)**2


def orthogonality_loss(psi, x, previous_states):
    if len(previous_states) == 0:
        return torch.tensor(0.0, device=device)

    loss = 0.0
    for psi_prev in previous_states:
        overlap = trapz_torch(psi * psi_prev, x)
        loss = loss + overlap**2
    return loss


def schrodinger_residual(model, x, potential_type):
    x.requires_grad_(True)
    psi = model(x, potential_type=potential_type)
    psi_xx = second_derivative(psi, x)
    Vx = potential(x, potential_type=potential_type)
    E = model.energy
    residual = -0.5 * psi_xx + Vx * psi - E * psi
    return residual, psi


## 6. Train one eigenstate

In [ ]:
def train_one_state(state_index, previous_states, potential_type='harmonic'):
    model = PINNWaveFunction(hidden_dim=64, n_hidden=4).to(device)

    if potential_type == 'box':
        L = x_max - x_min
        guess = (np.pi**2 * (state_index + 1)**2) / (2 * L**2)
    elif potential_type == 'harmonic':
        guess = state_index + 0.5
    elif potential_type == 'finite_well':
        guess = 0.5 + state_index
    else:
        guess = 1.0

    with torch.no_grad():
        model.energy[:] = torch.tensor([guess], dtype=torch.float32, device=device)

    optimizer = optim.Adam(model.parameters(), lr=LR)

    x_train = torch.linspace(x_min, x_max, N_collocation, device=device).view(-1, 1)
    loss_history = []
    energy_history = []

    for epoch in range(EPOCHS):
        optimizer.zero_grad()

        residual, psi = schrodinger_residual(model, x_train, potential_type)
        psi_normed = normalize_wavefunction(psi, x_train)

        loss_pde = torch.mean(residual**2)
        loss_bc = bc_loss(model, potential_type)
        loss_norm = normalization_loss(psi, x_train)
        loss_orth = orthogonality_loss(psi_normed, x_train, previous_states)

        loss = (
            loss_pde
            + LAMBDA_BC * loss_bc
            + LAMBDA_NORM * loss_norm
            + LAMBDA_ORTH * loss_orth
        )

        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        energy_history.append(model.energy.item())

        if epoch % 500 == 0 or epoch == EPOCHS - 1:
            print(
                f'State {state_index}, Epoch {epoch:5d}, '
                f'Loss={loss.item():.6e}, '
                f'E={model.energy.item():.6f}, '
                f'PDE={loss_pde.item():.3e}, '
                f'BC={loss_bc.item():.3e}, '
                f'Norm={loss_norm.item():.3e}, '
                f'Orth={loss_orth.item():.3e}'
            )

    x_dense = torch.linspace(x_min, x_max, N_plot, device=device).view(-1, 1)
    with torch.enable_grad():
        psi_dense = model(x_dense, potential_type=potential_type)
    psi_dense = normalize_wavefunction(psi_dense, x_dense).detach()

    return model, x_dense.detach(), psi_dense, np.array(loss_history), np.array(energy_history)


## 7. Train multiple eigenstates sequentially

In [ ]:
previous_states = []
trained_models = []
all_x = None
all_psi = []
all_E = []
all_loss_histories = []
all_energy_histories = []

for n in range(N_STATES):
    print(f'\nTraining state {n} for potential: {POTENTIAL_TYPE}')
    model, x_dense, psi_dense, loss_hist, energy_hist = train_one_state(
        n, previous_states, potential_type=POTENTIAL_TYPE
    )

    previous_states.append(psi_dense)
    trained_models.append(model)
    all_x = x_dense
    all_psi.append(psi_dense.cpu().numpy().flatten())
    all_E.append(model.energy.item())
    all_loss_histories.append(loss_hist)
    all_energy_histories.append(energy_hist)

print('\nFinal learned energies:')
for i, E in enumerate(all_E):
    print(f'State {i}: E = {E:.6f}')


## 8. Plot wavefunctions over the potential

In [ ]:
x_np = all_x.cpu().numpy().flatten()
V_np = potential(all_x, POTENTIAL_TYPE).detach().cpu().numpy().flatten()

plt.figure(figsize=(10, 6))
plt.plot(x_np, V_np, label='V(x)', linewidth=2)

for i, psi_np in enumerate(all_psi):
    plt.plot(x_np, psi_np + all_E[i], label=f'$\\psi_{i}(x) + E_{i}$')
    plt.axhline(all_E[i], linestyle='--', alpha=0.5)

plt.xlabel('x')
plt.ylabel('Energy / Wavefunction')
plt.title(f'PINN Schrödinger Solutions: {POTENTIAL_TYPE}')
plt.legend()
plt.grid(True)
plt.show()


## 9. Plot probability densities

In [ ]:
plt.figure(figsize=(10, 6))
for i, psi_np in enumerate(all_psi):
    plt.plot(x_np, psi_np**2, label=f'$|\\psi_{i}(x)|^2$')
plt.xlabel('x')
plt.ylabel('Probability density')
plt.title(f'Probability Densities: {POTENTIAL_TYPE}')
plt.legend()
plt.grid(True)
plt.show()


## 10. Plot energy levels

In [ ]:
plt.figure(figsize=(6, 6))
for i, E in enumerate(all_E):
    plt.hlines(E, xmin=0, xmax=1, linewidth=2, label=f'n={i}, E={E:.4f}')
plt.xlim(0, 1)
plt.xticks([])
plt.ylabel('Energy')
plt.title(f'Energy Levels: {POTENTIAL_TYPE}')
plt.legend()
plt.grid(True, axis='y')
plt.show()


## 11. Plot training loss

In [ ]:
plt.figure(figsize=(10, 6))
for i, hist in enumerate(all_loss_histories):
    plt.plot(hist, label=f'State {i}')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss History')
plt.legend()
plt.grid(True)
plt.show()


## 12. Plot energy convergence

In [ ]:
plt.figure(figsize=(10, 6))
for i, hist in enumerate(all_energy_histories):
    plt.plot(hist, label=f'State {i}')
plt.xlabel('Epoch')
plt.ylabel('Energy estimate')
plt.title('Energy Convergence')
plt.legend()
plt.grid(True)
plt.show()


## 13. Notes

- For the harmonic oscillator, expected energies are approximately `0.5, 1.5, 2.5, ...`
- For the particle in a box, the wavefunctions should be sinusoidal and vanish at the boundaries
- For the finite well, bound states should decay outside the well due to tunneling

You can improve the notebook further by adding:

- exact analytical comparisons
- fidelity calculations
- parity constraints
- finite-difference benchmarking
